# Text-to-SQL QLoRA on a free T4

Fine-tunes Qwen2.5-Coder-1.5B-Instruct on the execution-validated SQLite dataset built by the ETL
pipeline in this repo, then checks the result by *running* the generated SQL.

Run the cells in order and read the output of each before moving on. Runtime: **T4 GPU**.
Needs two Colab secrets: `HF_TOKEN` (write access) and `WANDB_API_KEY`.

## 0. Runtime check

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
gpu = torch.cuda.get_device_name(0)
assert "T4" in gpu, f"expected a T4, got {gpu}"
print("torch", torch.__version__, "| cuda", torch.version.cuda)

## 1. Install
This is Unsloth's Colab recipe for Qwen2.5-Coder; the transformers/trl pins matter.

In [ ]:
%%capture
import os
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2
!pip install wandb

In [ ]:
!pip freeze | grep -E "unsloth|trl|transformers|peft|bitsandbytes|torch|datasets"

## 2. Secrets

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
login(token=os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
print("logged in as", HF_USER)

## 3. Config
Same values as `configs/train.yaml`. The dataset repo comes from the ETL run (`make push-dataset`).

In [ ]:
import subprocess

CFG = dict(
    base_model="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    lora_r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    epochs=1, batch_size=8, grad_accum=2,
    learning_rate=2e-4, lr_scheduler="cosine", warmup_ratio=0.03,
    optimizer="adamw_8bit", weight_decay=0.01,
    eval_steps=100, save_steps=200, seed=3407,
)
DATASET_REPO = f"{HF_USER}/text2sql-sqlite-clean"
OUT_REPO_LORA = f"{HF_USER}/text2sql-qwen2.5-coder-1.5b-lora"
OUT_REPO_GGUF = f"{HF_USER}/text2sql-qwen2.5-coder-1.5b-GGUF"
REPO_URL = "https://github.com/wittyparth/text2sql-specialist"  # must be public for this clone

# the repo gives us the same prompt/schema/sqlexec code the eval and the app use
!rm -rf repo && git clone --depth 1 {REPO_URL} repo
GIT_SHA = subprocess.run(["git", "-C", "repo", "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
import sys; sys.path.insert(0, "repo/src")
print(DATASET_REPO, "|", OUT_REPO_LORA, "|", OUT_REPO_GGUF, "| repo", GIT_SHA)

## 4. Load the 4-bit base model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG["base_model"],
    max_seq_length=CFG["max_seq_length"],
    dtype=None,            # picks fp16 on a T4
    load_in_4bit=True,
)
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["seed"],
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")

## 6. Data
Each row already holds the final chat messages (system prompt, `Schema:` + question, SQL answer),
built by the same `build_messages` the evaluation and the app use.

In [ ]:
from datasets import load_dataset

train_ds = load_dataset(DATASET_REPO, "train", split="train")
val_ds = load_dataset(DATASET_REPO, "validation", split="validation")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

train_text = train_ds.map(to_text)
val_text = val_ds.map(to_text)

lengths = [len(tokenizer(t)["input_ids"]) for t in train_text["text"]]
print(f"{len(train_text)} train / {len(val_text)} val rows; tokens max {max(lengths)}, "
      f"mean {sum(lengths) / len(lengths):.0f}")
assert max(lengths) <= CFG["max_seq_length"]
print(train_text[0]["text"])

## 7. Baseline before training
Greedy-generate SQL for all 300 validation questions, run it in SQLite, and score it with the same
comparison rules as the evaluation harness. This is the "before" number for this notebook.

Batched generation can differ slightly from one-request-at-a-time serving, so treat this as a quick
check; the final numbers come from the eval harness on the shipped GGUF.

In [ ]:
from text2sql.prompt import extract_sql
from text2sql.schema import load_context
from text2sql.sqlexec import results_match, run_select

tokenizer.padding_side = "left"

def generate_sql(rows, batch_size=16, max_new_tokens=256):
    outs = []
    for i in range(0, len(rows), batch_size):
        prompts = [tokenizer.apply_chat_template(r["messages"][:2], tokenize=False,
                                                 add_generation_prompt=True)
                   for r in rows[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def execution_accuracy(rows, raw_outputs):
    hits, errors = [], 0
    for row, out in zip(rows, raw_outputs):
        meta = row["meta"]
        conn = load_context(meta["context_sqlite"])
        gold = run_select(conn, meta["gold_sql"])
        pred = run_select(conn, extract_sql(out))
        conn.close()
        if not pred.ok:
            errors += 1
        hits.append(pred.ok and gold.ok and results_match(pred.rows, gold.rows, meta["order_matters"]))
    return sum(hits) / len(hits), errors, hits

val_rows = list(val_ds)
FastLanguageModel.for_inference(model)
base_out = generate_sql(val_rows)
base_acc, base_errors, base_hits = execution_accuracy(val_rows, base_out)
print(f"BASE: execution accuracy {base_acc:.1%} on {len(val_rows)} val items, "
      f"{base_errors} did not execute")
for row, out in list(zip(val_rows, base_out))[:3]:
    print("\nQ:", row["messages"][1]["content"].split("Question:")[-1].strip())
    print("gold:", row["meta"]["gold_sql"]); print("base:", extract_sql(out))

## 8. Trainer

In [ ]:
from trl import SFTConfig, SFTTrainer

FastLanguageModel.for_training(model)

run_name = f"qwen2.5c-1.5b-qlora-r{CFG['lora_r']}-{GIT_SHA}"
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=val_text,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=CFG["max_seq_length"],
        packing=False,
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        num_train_epochs=CFG["epochs"],
        learning_rate=CFG["learning_rate"],
        lr_scheduler_type=CFG["lr_scheduler"],
        warmup_ratio=CFG["warmup_ratio"],
        optim=CFG["optimizer"],
        weight_decay=CFG["weight_decay"],
        fp16=True, bf16=False,                       # a T4 has no bf16
        logging_steps=10,
        eval_strategy="steps", eval_steps=CFG["eval_steps"],
        save_strategy="steps", save_steps=CFG["save_steps"], save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        push_to_hub=True, hub_model_id=OUT_REPO_LORA, hub_strategy="checkpoint",
        output_dir="outputs",
        seed=CFG["seed"],
        report_to="wandb", run_name=run_name,
    ),
)

## 9. Train on the answer only
Without this the model also learns to predict the schema and the question. Check the masking by
decoding exactly the tokens that still count towards the loss: it has to be the SQL plus `<|im_end|>`.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
labels = trainer.train_dataset[0]["labels"]
target = tokenizer.decode([t for t in labels if t != -100])
expected = train_ds[0]["messages"][-1]["content"] + "<|im_end|>"
print(repr(target))
assert target.strip() == expected, "loss mask is wrong, don't train"

## 10. Time projection
The callback below prints a projection of the whole run after 20 steps. If it says more than about
50 minutes, stop the cell and set `max_steps` in cell 8 so the run fits in 45 minutes (and say so in
the README). Measuring inside the real run avoids a throw-away probe that would train the model for 20
steps on a different schedule.

In [ ]:
import time
from transformers import TrainerCallback

class Projection(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if state.global_step == 20:
            per_step = (time.time() - self.t0) / 20
            print(f"\n>>> {per_step:.2f} s/step -> {per_step * state.max_steps / 60:.0f} min "
                  f"for {state.max_steps} steps")

trainer.add_callback(Projection())

## 11. Train
Checkpoints are pushed to the Hub every 200 steps. After a disconnect, reconnect, re-run cells 0-10
and use `trainer.train(resume_from_checkpoint=True)` instead.

In [ ]:
import wandb

wandb.init(project="text2sql-specialist", name=run_name, config={
    **CFG, "dataset": DATASET_REPO, "git_sha": GIT_SHA, "gpu": gpu,
    "trainable_params": trainable, "baseline_val_ex": base_acc,
})
stats = trainer.train()
print(stats.metrics)

## 12. Did it help?
Same 300 validation questions, same scoring as cell 7. This is the go/no-go before exporting.

In [ ]:
FastLanguageModel.for_inference(model)
ft_out = generate_sql(val_rows)
ft_acc, ft_errors, ft_hits = execution_accuracy(val_rows, ft_out)
print(f"BASE        {base_acc:.1%}  ({base_errors} did not execute)")
print(f"FINE-TUNED  {ft_acc:.1%}  ({ft_errors} did not execute)   n={len(val_rows)}")
wandb.run.summary.update({"val_ex_base": base_acc, "val_ex_finetuned": ft_acc})

table = wandb.Table(columns=["question", "gold", "base", "finetuned", "base_ok", "ft_ok"])
for row, b, f, bh, fh in list(zip(val_rows, base_out, ft_out, base_hits, ft_hits))[:20]:
    table.add_data(row["messages"][1]["content"].split("Question:")[-1].strip(),
                   row["meta"]["gold_sql"], extract_sql(b), extract_sql(f), bh, fh)
wandb.log({"val_predictions": table})

## 13. Save the LoRA adapter

In [ ]:
from huggingface_hub import ModelCard

model.push_to_hub(OUT_REPO_LORA, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(OUT_REPO_LORA, token=os.environ["HF_TOKEN"])
card = f'''---
base_model: Qwen/Qwen2.5-Coder-1.5B-Instruct
library_name: peft
license: apache-2.0
datasets: [{DATASET_REPO}]
tags: [text-to-sql, sqlite, qlora, unsloth]
---
# Qwen2.5-Coder-1.5B text-to-SQL (SQLite) LoRA

QLoRA adapter (r={CFG["lora_r"]}, alpha={CFG["lora_alpha"]}, all linear layers) trained for
{CFG["epochs"]} epoch on [{DATASET_REPO}](https://huggingface.co/datasets/{DATASET_REPO}).

Quick check on 300 validation questions from unseen domains (execution accuracy, notebook
batched greedy decoding): base {base_acc:.1%} -> fine-tuned {ft_acc:.1%}.
Final numbers come from the evaluation harness in the code repository ({REPO_URL}).
Git commit used for training: `{GIT_SHA}`. Training run: {wandb.run.url}
'''
ModelCard(card).push_to_hub(OUT_REPO_LORA, token=os.environ["HF_TOKEN"])

## 14. Export GGUF
Takes about ten minutes because llama.cpp is compiled first. If it fails, use the merged-16bit
fallback in `scripts/export_gguf_local.sh` and convert on a Mac.

In [ ]:
model.push_to_hub_gguf(
    OUT_REPO_GGUF, tokenizer,
    quantization_method=["q4_k_m", "q8_0", "f16"],
    token=os.environ["HF_TOKEN"],
)

## 15. Finish

In [ ]:
run_url = wandb.run.url
wandb.finish()
print("adapter :", f"https://huggingface.co/{OUT_REPO_LORA}")
print("gguf    :", f"https://huggingface.co/{OUT_REPO_GGUF}")
print("dataset :", f"https://huggingface.co/datasets/{DATASET_REPO}")
print("wandb   :", run_url)